In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

# ============================================================
# Parameters supplied by Databricks Asset Bundle
# ============================================================

dbutils.widgets.text("catalog", "dev_catalog")

CATALOG = dbutils.widgets.get("catalog")


# ============================================================
# Configuration
# ============================================================

SILVER_TABLE = (
    f"{CATALOG}.silver.customer"
)

GOLD_TABLE = (
    f"{CATALOG}.gold.customer_summary"
)

LOG_TABLE = (
    f"{CATALOG}.control.pipeline_run_log"
)

PIPELINE_NAME = "customer_etl_job"
TASK_NAME = "gold_customer"


# ============================================================
# Generate Run ID
# ============================================================

run_id = str(
    spark.sql("SELECT uuid()").first()[0]
)

start_timestamp = spark.sql(
    "SELECT current_timestamp()"
).first()[0]


# ============================================================
# Pipeline Log Schema
# ============================================================

log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("task_name", StringType(), True),
    StructField("start_timestamp", TimestampType(), True),
    StructField("end_timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("records_processed", LongType(), True),
    StructField("records_rejected", LongType(), True),
    StructField("error_message", StringType(), True)
])


# ============================================================
# Gold Processing
# ============================================================

try:

    print("Starting Gold customer aggregation...")

    print(f"Catalog: {CATALOG}")
    print(f"Silver Table: {SILVER_TABLE}")
    print(f"Gold Table: {GOLD_TABLE}")


    # ========================================================
    # Read Silver
    # ========================================================

    df = spark.table(SILVER_TABLE)


    # ========================================================
    # Create Gold Aggregation
    # ========================================================

    gold_df = (
        df
        .groupBy("City")
        .agg(
            F.countDistinct(
                "Customer_ID"
            ).alias(
                "Customer_Count"
            ),

            F.count(
                F.when(
                    F.col("Phone").isNotNull(),
                    F.col("Customer_ID")
                )
            ).alias(
                "Customers_With_Phone"
            )
        )
        .withColumn(
            "_last_updated_timestamp",
            F.current_timestamp()
        )
    )


    # ========================================================
    # Count Gold Records
    # ========================================================

    records_processed = gold_df.count()


    # ========================================================
    # Write Gold
    # ========================================================

    (
        gold_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_TABLE)
    )


    end_timestamp = spark.sql(
        "SELECT current_timestamp()"
    ).first()[0]


    # ========================================================
    # Write SUCCESS Log
    # ========================================================

    log_df = spark.createDataFrame(
        [(
            run_id,
            PIPELINE_NAME,
            TASK_NAME,
            start_timestamp,
            end_timestamp,
            "SUCCESS",
            records_processed,
            0,
            None
        )],
        schema=log_schema
    )

    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(LOG_TABLE)
    )


    print(
        "Gold customer aggregation completed successfully."
    )


except Exception as e:

    # ========================================================
    # Capture Failure
    # ========================================================

    end_timestamp = spark.sql(
        "SELECT current_timestamp()"
    ).first()[0]

    error_message = str(e)


    # ========================================================
    # Write FAILED Log
    # ========================================================

    log_df = spark.createDataFrame(
        [(
            run_id,
            PIPELINE_NAME,
            TASK_NAME,
            start_timestamp,
            end_timestamp,
            "FAILED",
            0,
            0,
            error_message
        )],
        schema=log_schema
    )

    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(LOG_TABLE)
    )


    print(
        f"Gold customer aggregation failed: {error_message}"
    )


    # ========================================================
    # Re-raise Exception
    # ========================================================

    raise